# Darukaa Adaptive Biodiversity Assessment — General Production Colab

**Purpose.** Run the generalized v1.1.0 assessment engine for a single polygon, KML/KMZ, multi-feature GeoJSON, or a Site Selection handoff containing multiple EMUs.

**Assessment unit:** EMU (Ecological Management Unit). An EMU may be multipart/disconnected. Each EMU is characterized, routed by ecological domain, benchmarked against its own compatible reference population, and retained as an auditable unit before project-level aggregation.

This notebook is intentionally **not ecosystem-specific**. Aquatic-only and terrestrial-only metrics are routed conditionally; mixed projects keep domain-specific reference populations separate.

## 0. Reproducibility and run controls

In [ ]:
REPO_URL = "https://github.com/G-auravSingh/reference-benchmarking.git"
GIT_REF = "main"  # Production/client runs must replace this with a verified commit SHA.
REPO_DIR = "/content/reference-benchmarking"
ADAPTIVE_DIR = f"{REPO_DIR}/darukaa_adaptive_v1.1.0"
GEE_PROJECT = "clever-grid-229708"
EXPECTED_PACKAGE_VERSION = "1.1.0"
OUTPUT_DIR = "/content/darukaa_adaptive_outputs"
DOMAIN_MODE = "auto"  # auto | terrestrial | aquatic | mixed
INPUT_MODE = "upload"  # upload | direct_path


In [ ]:
import os, shutil, subprocess, pathlib, sys, re
def run_cmd(cmd, cwd=None):
    print("$", " ".join(cmd)); r=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT); print(r.stdout); r.check_returncode(); return r.stdout.strip()

def clone_exact_ref(repo_url, git_ref, repo_dir):
    # git clone --branch accepts a branch/tag, not a raw commit SHA.
    git_ref = str(git_ref).strip()
    if not git_ref: raise ValueError("GIT_REF must be a branch, tag, or full commit SHA.")
    if os.path.exists(repo_dir): shutil.rmtree(repo_dir)
    is_sha = bool(re.fullmatch(r"[0-9a-fA-F]{40}", git_ref))
    run_cmd(["git", "clone", "--filter=blob:none", repo_url, repo_dir])
    if is_sha:
        run_cmd(["git", "-C", repo_dir, "fetch", "--depth", "1", "origin", git_ref])
        run_cmd(["git", "-C", repo_dir, "checkout", "--detach", "FETCH_HEAD"])
        actual = run_cmd(["git", "-C", repo_dir, "rev-parse", "HEAD"]).strip()
        if actual.lower() != git_ref.lower():
            raise RuntimeError(f"Checked-out commit {actual} does not match requested GIT_REF {git_ref}")
    else:
        run_cmd(["git", "-C", repo_dir, "checkout", git_ref])
    return run_cmd(["git", "-C", repo_dir, "rev-parse", "HEAD"]).strip()

ACTUAL_GIT_SHA = clone_exact_ref(REPO_URL, GIT_REF, REPO_DIR)
print("Pinned repository commit:", ACTUAL_GIT_SHA)
os.environ["DARUKAA_GIT_COMMIT"] = ACTUAL_GIT_SHA
sys.path.insert(0, ADAPTIVE_DIR)
subprocess.check_call([sys.executable,"-m","pip","install","-q","-e",ADAPTIVE_DIR])
import darukaa_adaptive
print("Package:",darukaa_adaptive.__version__)
assert darukaa_adaptive.__version__ == EXPECTED_PACKAGE_VERSION


## 1. Earth Engine initialization

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
print("Earth Engine initialized:", GEE_PROJECT)


## 2. Input acquisition

Accepted uploads:
- **Site Selection handoff ZIP** containing `tile_manifest.json` and referenced GeoJSON tiles;
- `tile_manifest.json` with its GeoJSON tiles;
- multi-feature `.geojson`;
- `.kml` / `.kmz`;
- single `.geojson` polygon.

For a production Site Selection run, use the formal handoff package rather than relying on filenames alone.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
if not uploaded: raise RuntimeError("No input uploaded.")
allowed={".zip",".json",".geojson",".kml",".kmz"}
inputs=[Path(x) for x in uploaded if Path(x).suffix.lower() in allowed]
if not inputs: raise ValueError(f"No supported input found: {list(uploaded)}")
INPUT_FILE=str(inputs[0])
print("Input:", INPUT_FILE)


## 3. Canonicalize the project into EMUs and inspect routing

In [ ]:
from darukaa_adaptive.inputs import load_project_input
project=load_project_input(INPUT_FILE, domain=DOMAIN_MODE)
print("Project:",project.project_id, project.project_name)
print("EMUs:",len(project.emus))
for e in project.emus:
    print(f"  {e.emu_id}: domain={e.resolved_domain(project.project_domain)}, area={e.area_ha:.3f} ha, parts={getattr(e.geometry,'geoms',[e.geometry]).__len__() if hasattr(e.geometry,'geoms') else 1}")


In [ ]:
assert project.validate()==[]
from darukaa_adaptive.handoff import validate_handoff_manifest
print("Input contract: PASS")


## 4. Load profile and configure output

In [ ]:
from darukaa_adaptive.config import AssessmentConfig
from pathlib import Path
from darukaa_adaptive.reference_profiles import ReferenceSelectionPolicy
profile_name = "aquatic_lake" if DOMAIN_MODE=="aquatic" else "terrestrial" if DOMAIN_MODE=="terrestrial" else "mixed"
PROFILE_PATH=f"{ADAPTIVE_DIR}/profiles/{profile_name}.yaml"
cfg=AssessmentConfig.from_yaml(PROFILE_PATH)
cfg.gee.project_id=GEE_PROJECT
cfg.output_dir=OUTPUT_DIR

# Stage C is intentionally manual and must be entered only after reviewing the
# reference diagnostics printed after a failed automatic reference attempt.
# Leave None for the first run. No KML/CSV reference fallback exists.
MANUAL_REFERENCE_HMI_THRESHOLD = None
cfg.reference.manual_hmi_threshold = MANUAL_REFERENCE_HMI_THRESHOLD

errors=cfg.validate()
if errors: raise ValueError("Configuration validation failed:\n- "+"\n- ".join(errors))
print("Profile:",cfg.profile.name,"version",cfg.profile.version)
print("Reference stages:", [x["stage"] for x in ReferenceSelectionPolicy(cfg).stage_plan()])
print("Manual HMI threshold:",cfg.reference.manual_hmi_threshold)


### 4A. Metric registry and scoring selection

The active v1.1.0 registry is the production inventory; the metric contract matrix is a historical audit/crosswalk and is not itself a list of metrics to score. Removed, diagnostic and context-only metrics cannot be promoted by overrides.

By default, active metrics classified as scoreable are included. Use `METRIC_OVERRIDES` only to demote a scoreable metric to contextual without changing its calculator.

**EII hierarchy:** the parent EII and its three provider-defined components are mutually exclusive in the headline condition score. The default is `parent`, which scores the published parent EII and keeps structural/compositional/functional components diagnostic. Use `components` only when intentionally replacing the parent with its components. `none` keeps all EII layers contextual.

**MSA:** the available `msa` candidate remains contextual/pending provenance; it is not scored merely because an asset identifier exists. Do not substitute a Europe-only CLC-based method for Indian sites.


In [ ]:
import pandas as pd
from darukaa_adaptive.registry import metric_selection_table, metric_scoreability, get_indicator_spec

# ---------------------------------------------------------------------------
# MANUAL SCORING CONTROL
# ---------------------------------------------------------------------------
# By default, every scientifically scoreable metric is scored. To demote a
# metric to contextual reporting, add it here without changing its calculator.
# Example:
#     METRIC_OVERRIDES = {"ndvi": "contextual", "habitat_health": "contextual"}
# Allowed values: "scored" or "contextual".
# Hard diagnostic/context-only/removed metrics cannot be promoted to scored.
METRIC_OVERRIDES = {}

# EII is a hard scientific hierarchy gate:
#   components -> structural + compositional + functional are scored
#   parent     -> parent EII is scored
#   none       -> all EII layers are contextual
EII_MODE = "parent"

# Validate manual controls before applying them so typos cannot silently change
# the scoring set.
_all_specs = {row["metric"]: row for row in metric_selection_table()}
_invalid_names = sorted(set(METRIC_OVERRIDES) - set(_all_specs))
if _invalid_names:
    raise ValueError(f"Unknown metric(s) in METRIC_OVERRIDES: {_invalid_names}")
_invalid_values = {m:v for m,v in METRIC_OVERRIDES.items() if v not in {"scored", "contextual"}}
if _invalid_values:
    raise ValueError(f"METRIC_OVERRIDES values must be 'scored' or 'contextual': {_invalid_values}")
if EII_MODE not in {"components", "parent", "none"}:
    raise ValueError("EII_MODE must be 'components', 'parent', or 'none'")

_hard_not_scoreable = {m for m,row in _all_specs.items() if row["scoreability"] != "default_scored"}
_hierarchy_excluded = {"eii"} if EII_MODE != "parent" else set()
_hierarchy_excluded |= {"eii_structural", "eii_compositional", "eii_functional"} if EII_MODE != "components" else set()
_promotion_attempts = {
    m:v for m,v in METRIC_OVERRIDES.items()
    if v == "scored" and (m in _hard_not_scoreable or m in _hierarchy_excluded)
}
if _promotion_attempts:
    raise ValueError(
        "Cannot promote hard-gated metrics to scored. "
        f"Remove these overrides or change the EII_MODE: {_promotion_attempts}"
    )

cfg.scoring.score_all_scoreable_metrics = True
cfg.scoring.metric_overrides = METRIC_OVERRIDES
cfg.scoring.eii_mode = EII_MODE

metric_registry_df = pd.DataFrame(metric_selection_table(cfg))
from darukaa_adaptive.registry import get_indicator_spec, effective_scoring_role
_role_map = {"SCORED":"scored", "PRESSURE":"scored", "CONTEXTUAL":"contextual", "DIAGNOSTIC":"diagnostic", "REMOVED":"removed"}
metric_registry_df["runtime_selection"] = [
    _role_map.get(effective_scoring_role(get_indicator_spec(name), cfg), "contextual")
    for name in metric_registry_df["metric"]
]

# Apply the same EII hierarchy and manual overrides used by the runtime scorer.
metric_registry_df.loc[metric_registry_df.metric.eq("eii"), "runtime_selection"] = (
    "scored" if EII_MODE == "parent" else "contextual"
)
metric_registry_df.loc[metric_registry_df.metric.isin(
    ["eii_structural", "eii_compositional", "eii_functional"]
), "runtime_selection"] = (
    "scored" if EII_MODE == "components" else "contextual"
)
for metric_name, mode in METRIC_OVERRIDES.items():
    metric_registry_df.loc[metric_registry_df.metric.eq(metric_name), "runtime_selection"] = mode

print(f"Registered metrics: {len(metric_registry_df)}")
print(f"Default scoreable metrics: {(metric_registry_df.scoreability == 'default_scored').sum()}")
print(f"Runtime scored metrics: {(metric_registry_df.runtime_selection == 'scored').sum()}")
print(f"Contextual/diagnostic metrics: {(metric_registry_df.runtime_selection != 'scored').sum()}")
print(f"EII scoring mode: {EII_MODE}")
print("Manual overrides:", METRIC_OVERRIDES or "none")

display(metric_registry_df[[
    "metric", "display_name", "pillar", "construct", "domain", "direction",
    "scoreability", "registry_role", "runtime_selection", "reference_type", "source",
    "dataset_asset", "native_scale_m", "spatial_resolution", "temporal_resolution", "temporal_period", "notes"
]])



## 5. Run the complete EMU → reference → benchmark → scoring → reporting pipeline

In [ ]:
from darukaa_adaptive.pipeline import AdaptivePipeline
runner=AdaptivePipeline(cfg)
result=runner.run(INPUT_FILE, domain=DOMAIN_MODE)
print("Completed EMUs:",len(result["emus"]))
print("Output directory:",result["output_dir"])
print("Project overall:",result["project_overall"])
print("\nEMU comparison:")
print(result["emu_comparison"].to_string(index=False) if not result["emu_comparison"].empty else "No scored EMU comparison available.")
print("Output archive:", result.get("output_archive"))
print("Project output QA:", result.get("project_output_qa", {}))

# Automatically download the complete project output bundle.
# The archive is created by the pipeline outside the output directory, so it
# contains every EMU folder, all CSV/JSON outputs, and the HTML report.
from google.colab import files
files.download(result["output_archive"])


## 6. Reference governance and coverage QA

In [ ]:
for r in result["emus"]:
    print("\nEMU",r["emu_id"],"domain",r["emu_domain"])
    print("  area_ha:",round(r["emu_area_ha"],3))
    print("  reference populations:", list(r["reference_populations"]))
    for k,v in r["reference_populations"].items():
        print("  ",k,"status=",v.get("status"),"approval=",v.get("approval"),"state=",v.get("reference_state"))


## 7. Deliverables and audit record

The run directory contains one auditable output tree per EMU plus project-level aggregation tables, concern-band distribution, automated output QA, and `project_assessment_manifest.json`. The project HTML report presents condition separately from anthropogenic pressure, includes project-level pillar/metric summaries, EMU comparison, a condition concern distribution, evidence gaps, QA flags and an EMU output/reference index. The concern distribution uses existing score bands and assigns an overall EMU band only when P1, P2 and P3 are all scored; partial and insufficient EMUs remain explicit. Area shares assume EMUs do not overlap.

The report is a generated assessment summary, not a substitute for reviewing metric-level CSV/JSON records, reference QA, provenance and scientific limitations.


In [ ]:
from pathlib import Path
out=Path(result["output_dir"])
for name in ["project_assessment_manifest.json","project_overall_scorecard.json","project_pillar_aggregation.csv","project_metric_raw_aggregation.csv","project_metric_score_aggregation.csv","emu_ecological_comparison.csv","emu_condition_concern_distribution.csv","project_output_qa.json","Project_Biodiversity_Baseline_Report.html"]:
    print(name, "->", (out/name), "exists:", (out/name).exists())


## 8. Production release checklist

Before client delivery: pin the exact Git commit; retain the input/handoff archive; retain `project_assessment_manifest.json`; inspect reference governance for every EMU; confirm metric coverage and missing-data status; review proxy labels, uncertainty and MSA provenance; verify concern-band counts/area shares against the CSV; check the output-QA flags; verify report tables against CSV outputs; and archive the exact configuration/profile used. Do not interpret an automated QA PASS as ecological validation.
